# Color image processing

Inspect RGB channels, mix colors, apply pseudocolor, and adjust intensity without confusing RGB with OpenCV BGR ordering.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src/image_processing").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from image_processing import load_sample, show_images

try:
    import cv2 as cv
except ImportError:
    cv = None
    print("OpenCV is unavailable. NumPy examples still run; install requirements.txt for comparisons.")

## RGB channels

The bundled color sample is loaded in RGB order.

In [ ]:
rgb = load_sample("color")
show_images([rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]], ["Red", "Green", "Blue"])

## Mixing colors

A weighted sum mixes red with white. The weights sum to one.

In [ ]:
red = np.array([255, 0, 0], dtype=float)
white = np.array([255, 255, 255], dtype=float)
swatches = [np.tile((weight * red + (1 - weight) * white).astype(np.uint8), (30, 30, 1)) for weight in [1, .75, .5, .25, 0]]
show_images(swatches, ["Red", "75% red", "50% red", "25% red", "White"], gray=False)

## Grayscale and BGR ordering

OpenCV imread returns BGR. Convert it before displaying with Matplotlib.

In [ ]:
if cv is not None:
    bgr = cv.imread(str(ROOT / "data/sample_images/color.png"))
    converted = cv.cvtColor(bgr, cv.COLOR_BGR2RGB)
    np.testing.assert_array_equal(converted, rgb)
    gray = cv.cvtColor(rgb, cv.COLOR_RGB2GRAY)
    show_images([rgb, gray], ["RGB", "Grayscale"], gray=False)

## Pseudocolor

Assign colors to intensity intervals. These colors visualize intensity; they do not reconstruct the original scene colors.

In [ ]:
gray = load_sample()
pseudo = np.repeat(gray[:, :, None], 3, axis=2)
pseudo[(gray >= 60) & (gray < 100)] = [0, 0, 255]
pseudo[(gray >= 100) & (gray < 140)] = [0, 255, 0]
pseudo[(gray >= 140) & (gray < 180)] = [255, 255, 0]
pseudo[gray >= 180] = [255, 0, 0]
show_images([gray, pseudo], ["Grayscale", "Pseudocolor"], gray=False)

## Intensity scaling

Compute in floating point and clip before converting back to uint8.

In [ ]:
darker = np.clip(rgb.astype(float) * .7, 0, 255).astype(np.uint8)
show_images([rgb, darker], ["Original", "Scaled"], gray=False)